In [13]:
# EDA
import pandas as pd
import plotly.express as px
import numpy as np

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, ElasticNet, HuberRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import StackingRegressor
from sklearn.metrics import root_mean_squared_error, r2_score

In [14]:
# Carregar os dados já tratados
df_costs = pd.read_csv('./datasets/healthcosts_cleaned.csv')

In [15]:
# Mostrar as primeiras linhas
df_costs.head(20)

,age,sex,bmi,children,smoker,region,medical charges
0,19,female,27.900,0,1,southwest,16884.92400
1,18,male,33.770,1,0,southeast,1725.55230
2,28,male,33.000,3,0,southeast,4449.46200
3,33,male,22.705,0,0,northwest,21984.47061
4,32,male,28.880,0,0,northwest,3866.85520
5,31,female,25.740,0,0,southeast,3756.62160
6,46,female,33.440,1,0,southeast,8240.58960
7,37,female,27.740,3,0,northwest,7281.50560
8,37,male,29.830,2,0,northeast,6406.41070
9,60,female,25.840,0,0,northwest,28923.13692


In [16]:
# Mostrar as ultimas linhas
df_costs.tail(20)

,age,sex,bmi,children,smoker,region,medical charges
1318,35,male,39.710,4,0,northeast,19496.71917
1319,39,female,26.315,2,0,northwest,7201.70085
1320,31,male,31.065,3,0,northwest,5425.02335
1321,62,male,26.695,0,1,northeast,28101.33305
1322,62,male,38.830,0,0,southeast,12981.34570
1323,42,female,40.370,2,1,southeast,43896.37630
1324,31,male,25.935,1,0,northwest,4239.89265
1325,61,male,33.535,0,0,northeast,13143.33665
1326,42,female,32.870,0,0,northeast,7050.02130
1327,51,male,30.030,1,0,southeast,9377.90470


In [17]:
# Mostrar a estrutura do dataset
df_costs.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   age              1338 non-null   int64  
 1   sex              1338 non-null   object 
 2   bmi              1338 non-null   float64
 3   children         1338 non-null   int64  
 4   smoker           1338 non-null   int64  
 5   region           1338 non-null   object 
 6   medical charges  1338 non-null   float64
dtypes: float64(2), int64(3), object(2)
memory usage: 73.3+ KB


# Preparação dos dados

In [18]:
# Preparar os dados para o modelo
X = df_costs.drop(columns=['medical charges'])
y = df_costs['medical charges']

In [19]:
# Carregar preprocessor
import joblib
preprocessor = joblib.load('./preprocessor_dataset_healthcosts.pkl')

In [20]:
# Dividir o dataset entre treinamento e teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=51)

In [21]:
# Aplicar o preprocessor em treinamento e teste
# Treinamento = fit & transform
# Teste = transform, considerando o treinamento que foi feito anteriormente
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

In [22]:
# Mostrar as dimensoes dos conjuntos
print(f'Treinamento: {X_train.shape}')
print(f'Teste: {X_test.shape}')

Treinamento: (1070, 10)
Teste: (268, 10)


# Treinamento do Modelo Stacking

In [26]:
# Criar o modelo de Stacking Regressor Vanilla

# Algoritmos base
lr_model_vanilla = LinearRegression()
elastic_model_vanilla = ElasticNet(alpha=1.0, l1_ratio=0.5, random_state=51)
tree_model_vanilla = DecisionTreeRegressor(random_state=51)

# Meta-modelo ou Meta-learner
huber_model_vanilla = HuberRegressor()

stacking_model_vanilla = StackingRegressor(
    estimators = [
        ('linear regression', lr_model_vanilla),
        ('elastic', elastic_model_vanilla),
        ('decision tree', tree_model_vanilla)
    ],
    final_estimator=huber_model_vanilla,
    # Passthrough = False -> Usa apenas as predições dos estimadores base
    # Passthrough = True -> Usa as predições dos estimadores base + conjunto de treinamento (dataset)
    passthrough=False
)

In [ ]:
# Criar o modelo de Stacking Regressor Blending

# Algoritmos base
lr_model_blending = LinearRegression()
elastic_model_blending = ElasticNet(alpha=1.0, l1_ratio=0.5, random_state=51)
tree_model_blending = DecisionTreeRegressor(random_state=51)

# Meta-modelo ou Meta-learner
huber_model_blending = HuberRegressor()

stacking_model_blending = StackingRegressor(
    estimators = [
        ('linear regression', lr_model_blending),
        ('elastic', elastic_model_blending),
        ('decision tree', tree_model_blending)
    ],
    final_estimator=huber_model_blending,
    # Passthrough = False -> Usa apenas as predições dos estimadores base
    # Passthrough = True -> Usa as predições dos estimadores base + conjunto de treinamento (dataset)
    passthrough=True
)

In [28]:
# Treinar o modelo
stacking_model_vanilla.fit(X_train, y_train)

StackingRegressor(estimators=[('linear regression', LinearRegression()),
                              ('elastic', ElasticNet(random_state=51)),
                              ('decision tree',
                               DecisionTreeRegressor(random_state=51))],
                  final_estimator=HuberRegressor())

In [29]:
# Treinar o modelo
stacking_model_blending.fit(X_train, y_train)

StackingRegressor(estimators=[('linear regression', LinearRegression()),
                              ('elastic', ElasticNet(random_state=51)),
                              ('decision tree',
                               DecisionTreeRegressor(random_state=51))],
                  final_estimator=HuberRegressor(), passthrough=True)